# EP09｜CUDA 是什麼？輝達最強的不是晶片

像素腦底加！這是影片裡的實測：同一個矩陣乘法，「自己寫」和「最佳化過的函式庫」差多少，還有只改記憶體讀取的順序會差多少。按上方的 **執行階段 → 全部執行**（或每格按 ▶️）就能自己跑一遍。

⚠️ 這本筆記本的實驗**全部在 CPU 上做**，用來示意 CUDA 和函式庫背後的道理（自己寫的核心函式 vs cuBLAS、合併存取），不是 GPU 實測。所有寫法都限制只用 1 個執行緒，公平比較；每台電腦的秒數都不一樣，你的數字不會和影片完全相同。

📺 影片：[youtube.com/@pixelbraintw](https://www.youtube.com/@pixelbraintw)

## 0. 準備
先限制只用 1 個執行緒（一定要在 import numpy 之前設定），再載入 NumPy 和 Numba。Numba 會把 Python 迴圈編譯成機器碼。

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"; os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"; os.environ["NUMBA_NUM_THREADS"] = "1"
import time
import numpy as np
import numba
from numba import njit
print("NumPy", np.__version__, "Numba", numba.__version__)

def best(f, n=3):
    """跑 n 次，取最快的一次（秒）。"""
    ts = []
    for _ in range(n):
        t = time.perf_counter(); f(); ts.append(time.perf_counter() - t)
    return min(ts)

NumPy 2.5.3 Numba 0.68.1


## 1. 三種寫法算同一個矩陣乘法（256 × 256）
Python 一個一個算 → Numba 編譯同樣的迴圈（像自己寫一個核心函式）→ 最佳化函式庫（NumPy 底下的 BLAS）。

In [2]:
def py_loop(A, B):
    n = len(A); C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            s = 0.0
            for k in range(n):
                s += A[i][k] * B[k][j]
            C[i][j] = s
    return C

@njit(cache=False)
def nb_ijk(A, B):              # 自己寫、編譯過的樸素迴圈（像自己寫的 kernel）
    n = A.shape[0]; C = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            s = 0.0
            for k in range(n):
                s += A[i, k] * B[k, j]   # B 一直跳行讀：記憶體不連續
            C[i, j] = s
    return C

@njit(cache=False)
def nb_ikj(A, B):              # 只換迴圈順序：B 改成沿著一列連續讀
    n = A.shape[0]; C = np.zeros((n, n))
    for i in range(n):
        for k in range(n):
            a = A[i, k]
            for j in range(n):
                C[i, j] += a * B[k, j]
    return C

@njit(cache=False)
def nb_tiled(A, B, T):         # 再切成小塊，讓資料留在快取裡重複用
    n = A.shape[0]; C = np.zeros((n, n))
    for ii in range(0, n, T):
        for kk in range(0, n, T):
            for jj in range(0, n, T):
                for i in range(ii, min(ii + T, n)):
                    for k in range(kk, min(kk + T, n)):
                        a = A[i, k]
                        for j in range(jj, min(jj + T, n)):
                            C[i, j] += a * B[k, j]
    return C

In [3]:
rng = np.random.default_rng(0)
n1 = 256
A, B = rng.random((n1, n1)), rng.random((n1, n1))
ref = A @ B
Al, Bl = A.tolist(), B.tolist()
assert np.allclose(np.array(py_loop(Al, Bl)), ref)
for f in (nb_ijk, nb_ikj): assert np.allclose(f(A, B), ref)   # 第一次呼叫會先編譯
t_py = best(lambda: py_loop(Al, Bl), 2)
t_nb = best(lambda: nb_ijk(A, B), 5)
t_blas = best(lambda: A @ B, 30)
print(f"Python 一個一個算：{t_py:.3f} 秒")
print(f"Numba 編譯版：    {t_nb:.4f} 秒（快 {t_py / t_nb:.0f} 倍）")
print(f"最佳化函式庫：    {t_blas:.5f} 秒（再快 {t_nb / t_blas:.0f} 倍，比 Python 快 {t_py / t_blas:.0f} 倍）")

Python 一個一個算：0.563 秒
Numba 編譯版：    0.0143 秒（快 39 倍）
最佳化函式庫：    0.00052 秒（再快 28 倍，比 Python 快 1093 倍）


## 2. 我想追上函式庫（1024 × 1024）
原本的寫法 → 只換迴圈順序 → 再切成 64 × 64 小塊 → 函式庫。試試把 `T` 改成 16、32、128，看看哪個最快。

In [4]:
n2 = 1024
A2, B2 = rng.random((n2, n2)), rng.random((n2, n2))
ref2 = A2 @ B2
T = 64
for f in (nb_ijk, nb_ikj): assert np.allclose(f(A2, B2), ref2)
assert np.allclose(nb_tiled(A2, B2, T), ref2)
t_ijk = best(lambda: nb_ijk(A2, B2), 2)
t_ikj = best(lambda: nb_ikj(A2, B2), 3)
t_tile = best(lambda: nb_tiled(A2, B2, T), 3)
t_lib = best(lambda: A2 @ B2, 10)
print(f"原本的寫法：      {t_ijk:.2f} 秒")
print(f"只換迴圈順序：    {t_ikj:.3f} 秒（快 {t_ijk / t_ikj:.0f} 倍）")
print(f"切成 {T} × {T} 小塊：{t_tile:.3f} 秒")
print(f"函式庫：          {t_lib:.4f} 秒（比換順序的版本快 {t_ikj / t_lib:.0f} 倍）")

原本的寫法：      7.36 秒
只換迴圈順序：    0.359 秒（快 20 倍）
切成 64 × 64 小塊：0.662 秒
函式庫：          0.0297 秒（比換順序的版本快 12 倍）


## 3. 只改讀的順序（8192 × 8192，6,700 多萬個數字加總）
照記憶體順序橫著讀，和直著跳著讀。答案一樣，差在讀的順序。

In [5]:
@njit(cache=False)
def sum_rows(X):
    s = 0.0
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            s += X[i, j]
    return s

@njit(cache=False)
def sum_cols(X):
    s = 0.0
    for j in range(X.shape[1]):
        for i in range(X.shape[0]):
            s += X[i, j]
    return s

n3 = 8192
X = rng.random((n3, n3)).astype(np.float32)
a, b = sum_rows(X), sum_cols(X)
assert abs(a - b) / a < 1e-6
t_rows = best(lambda: sum_rows(X), 5)
t_cols = best(lambda: sum_cols(X), 3)
print(f"照順序橫著讀：{t_rows:.3f} 秒")
print(f"直著跳著讀：  {t_cols:.3f} 秒（慢 {t_cols / t_rows:.1f} 倍）")

照順序橫著讀：0.115 秒
直著跳著讀：  0.803 秒（慢 7.0 倍）


## 參考資料
- NVIDIA CUDA C++ Programming Guide：https://docs.nvidia.com/cuda/cuda-c-programming-guide/
- NVIDIA CUDA C++ Best Practices Guide（合併存取）：https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html
- cuBLAS：https://docs.nvidia.com/cuda/cublas/index.html
- cuDNN：https://developer.nvidia.com/cudnn
- AMD HIPIFY：https://rocm.docs.amd.com/projects/HIPIFY/en/latest/index.html
- Goto & van de Geijn, Anatomy of High-Performance Matrix Multiplication：https://cs.utexas.edu/~flame/pubs/GotoTOMS_final.pdf